In [3]:
!pip install transformers evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("eriktks/conll2003", revision="convert/parquet")

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [5]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [6]:
raw_datasets['train'][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [7]:
raw_datasets['train'][0]['tokens']

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

In [8]:
ner_features = raw_datasets['train'].features['ner_tags']

In [9]:
ner_features

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

In [10]:
label_names = ner_features.feature.names

In [11]:
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [12]:
words = raw_datasets["train"][0]["tokens"]
labels = raw_datasets["train"][0]["ner_tags"]
line1 = ""
line2 = ""

for word,label in zip(words,labels):
  full_label=label_names[label]
  max_length= max(len(word),len(full_label))
  line1 += word + " " * (max_length - len(word) + 1)
  line2 += full_label + " "*(max_length-len(full_label)+1)

print(line1)
print(line2)

EU    rejects German call to boycott British lamb . 
B-ORG O       B-MISC O    O  O       B-MISC  O    O 


In [13]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [14]:
pos_labels = raw_datasets['train'].features['pos_tags'].feature.names
chunk_labels = raw_datasets['train'].features['chunk_tags'].feature.names

In [15]:
# now lets see everything for index 4
print(raw_datasets['train'][4]['tokens'])

['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']


In [16]:
index_4_tokens = raw_datasets['train'][4]['tokens']
index_4_ners = raw_datasets['train'][4]['ner_tags']
index_4_poss = raw_datasets['train'][4]['pos_tags']
index_4_chunks = raw_datasets['train'][4]['chunk_tags']

print(f"{'TOKEN':<15} {'NER_TAG':<15} {'POS_TAG':<15} {'CHUNK_TAG'}")

for t, n, p, c in zip(index_4_tokens, index_4_ners, index_4_poss, index_4_chunks):
    ner_name = label_names[n]
    pos_name = pos_labels[p]
    chunk_name = chunk_labels[c]

    print(f"{t:<15} {ner_name:<15} {pos_name:<15} {chunk_name}")

TOKEN           NER_TAG         POS_TAG         CHUNK_TAG
Germany         B-LOC           NNP             B-NP
's              O               POS             B-NP
representative  O               NN              I-NP
to              O               TO              B-PP
the             O               DT              B-NP
European        B-ORG           NNP             I-NP
Union           I-ORG           NNP             I-NP
's              O               POS             B-NP
veterinary      O               JJ              I-NP
committee       O               NN              I-NP
Werner          B-PER           NNP             I-NP
Zwingmann       I-PER           NNP             I-NP
said            O               VBD             B-VP
on              O               IN              B-PP
Wednesday       O               NNP             B-NP
consumers       O               NNS             I-NP
should          O               MD              B-VP
buy             O               VB       

In [17]:
from transformers import AutoTokenizer
checkpoint="bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
tokenizer.is_fast

True

In [19]:
input = tokenizer(raw_datasets['train'][0]['tokens'],is_split_into_words=True)

In [20]:
input

{'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [21]:
print(f"{"type":>15} {"length":>15}")
print(f"{"token length":>15} {len(raw_datasets['train'][0]['tokens']):>15}")
print(f"{"input id":>15} {len(input['input_ids']):>15}")
print(f"{"token_type_ids":>15} {len(input['token_type_ids']):>15}")
print(f"{"attention_masks":>15} {len(input['attention_mask']):>15}")

           type          length
   token length               9
       input id              11
 token_type_ids              11
attention_masks              11


In [22]:
input.tokens()

['[CLS]',
 'eu',
 'rejects',
 'german',
 'call',
 'to',
 'boycott',
 'british',
 'lamb',
 '.',
 '[SEP]']

In [23]:
input.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, None]

In [24]:
## we have great problem , called token label alignment problem , as shape of tokens and labels i.e ner tags must match
##, but we will do tokenization then
## certain words will be broken down again into subwords , so what ner tags they should be provided

In [35]:
def align_labels_with_tokens(labels,word_ids):
  new_labels=[]
  current_word=None
  for word_id in word_ids:
    if word_id != current_word:
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)
    elif word_id is None:
      new_labels.append(-100)
    else:
      label=labels[word_id]

      if label% 2 == 1:
        label += 1
      new_labels.append(label)
  return new_labels

In [38]:
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(examples["tokens"],truncation=True, is_split_into_words=True)
  all_labels = examples["ner_tags"]
  new_labels=[]
  for i,label in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(label,word_ids))

  tokenized_inputs["labels"]=new_labels
  return tokenized_inputs




In [39]:
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels,
                                      batched=True,
                                      remove_columns=raw_datasets["train"].column_names)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [40]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [41]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1e66ba3610752f730cb574b2d58c9ea5d54b1c12fa4d1ffb0b5a11effb2adac3
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [42]:
import evaluate
metric = evaluate.load('seqeval')

In [53]:
import numpy as np

def compute_metrics(eval_preds):

    logits, labels = eval_preds

    # Correct axis
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [54]:
id2label = {i:label for i,label in enumerate(label_names)}
label2id = {v:k for k,v in id2label.items()}

In [45]:
from transformers import AutoModelForTokenClassification
model_checkpoint=checkpoint
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [46]:
model.config.num_labels

9

In [47]:
from huggingface_hub import notebook_login
notebook_login()

In [55]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

In [56]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    processing_class = tokenizer,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.023228,0.063221,0.931498,0.945136,0.938267,0.986385
2,0.019645,0.060336,0.935719,0.950522,0.943062,0.987450
3,0.011263,0.060636,0.937521,0.952036,0.944723,0.987783


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5268, training_loss=0.021328050391458613, metrics={'train_runtime': 557.1777, 'train_samples_per_second': 75.601, 'train_steps_per_second': 9.455, 'total_flos': 891900038010780.0, 'train_loss': 0.021328050391458613, 'epoch': 3.0})

In [58]:
trainer.push_to_hub(commit_message="Training complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/baiju0110/bert-finetuned-ner/commit/c6b4ed84f7f633c7a77325d824a63f19275a31c4', commit_message='Training complete', commit_description='', oid='c6b4ed84f7f633c7a77325d824a63f19275a31c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/baiju0110/bert-finetuned-ner', endpoint='https://huggingface.co', repo_type='model', repo_id='baiju0110/bert-finetuned-ner'), pr_revision=None, pr_num=None)